In [ ]:
!pip install pytorch-lightning
!pip install gdown

In [ ]:
import gdown

folder_id = "1IBMYahLwJ5pY9b-f4gCm_LZPCcYccH6h"

gdown.download_folder(
    id=folder_id,
    quiet=False,
    use_cookies=False
)

In [ ]:
!pip install scipy

In [ ]:
from scipy.io import loadmat
import numpy as np
import pandas as pd
import scipy.io
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [ ]:
sampling_freq = 128
FOCUSED_CLASS = 0
UNFOCUSED_CLASS = 1
DROWNSY_CLASS = 2

In [ ]:
path = "/content/EEG Data"

In [ ]:
import os

files = os.listdir(path)
len(files)

In [ ]:
columns = [
    'ED_COUNTER',    'ED_INTERPOLATED',    'ED_RAW_CQ',    'ED_AF3',    'ED_F7',
    'ED_F3',    'ED_FC5',    'ED_T7',    'ED_P7',    'ED_O1',
    'ED_O2',    'ED_P8',    'ED_T8',    'ED_FC6',    'ED_F4',
    'ED_F8',    'ED_AF4',    'ED_GYROX',    'ED_GYROY',    'ED_TIMESTAMP',
    'ED_ES_TIMESTAMP',    'ED_FUNC_ID',    'ED_FUNC_VALUE',    'ED_MARKER',    'ED_SYNC_SIGNAL'
]

In [ ]:
def get_state(timestamp):
    if timestamp <= 10*128*60:
        return FOCUSED_CLASS
    elif timestamp > 20*128*60:
        return UNFOCUSED_CLASS
    else:
        return DROWNSY_CLASS

# Scale data
scaler = StandardScaler()

In [ ]:
def get_EEG_data(data_root, filename):
    hz = sampling_freq
    mat = scipy.io.loadmat(data_root +"/"+ filename)
    data = mat["o"]["data"][0,0]
    eeg_df = pd.DataFrame(data, columns=columns)
    eeg_df = eeg_df.filter(['ED_AF3', 'ED_F7', 'ED_F3', 'ED_FC5',
                            'ED_T7', 'ED_P7', 'ED_O1', 'ED_O2',
                            'ED_P8', 'ED_T8', 'ED_FC6', 'ED_F4',
                            'ED_F8', 'ED_AF4'])
    labels = ['AF3','F7', 'F3','FC5','T7','P7','O1','O2','P8','T8', 'FC6','F4','F8','AF4']
    eeg_df.columns = labels
    eeg_df = pd.DataFrame(scaler.fit_transform(eeg_df), columns=eeg_df.columns)
    eeg_df.reset_index(inplace=True)
    eeg_df.rename(columns={'index': 'timestamp'}, inplace=True)

    eeg_df['state'] = eeg_df['timestamp'].apply(get_state)

    return eeg_df

In [ ]:
dataset = []
# For each file, print # minutes of data
for filename in files:
    data = get_EEG_data(path, filename)
    dataset.append(data)

In [ ]:
def split_epochs(data, hz, epoch_length=2, step_size=0.125):
  step = int(epoch_length * hz - step_size * hz)
  offset = int(epoch_length * hz)
  starts = []
  current = 0

# Generate the first series
  while current + offset <= data.shape[0]:
    starts.append(current)
    current += step

  # Generate the second series using a list comprehension
  ends = [x + offset for x in starts]

  # Lưu trữ các epoch
  epochs = []

  # Cắt các epoch từ tín hiệu
  for i in range(len(starts)):
    epoch = data.iloc[starts[i]:ends[i]]
    epochs.append(epoch)

  return epochs

In [ ]:
epochs_data = []
for eeg in dataset:
  epochs = split_epochs(eeg, sampling_freq)
  for epoch in epochs:  # Iterate directly over the epochs
    epochs_data.append(epoch)  # Append each DataFrame to the list

In [ ]:
from torch.utils.data import Dataset # Added this import

class EEGDataset(Dataset):
  def __init__(self, dataframes, target_column='state', wavelet='db6', level=4):
    self.data = []
    self.targets = []
    self.scaler = StandardScaler()

    print(f"Processing {len(dataframes)} dataframes...")

    for df in dataframes:
      # Extract target
      self.targets.append(df[target_column].mode()[0])

      # Process features
      feature = df.drop(columns=[target_column, 'timestamp'], errors='ignore')
      self.data.append(feature.values)

    # Convert lists to tensors AFTER the loop
    self.data = torch.tensor(self.data, dtype=torch.float32)
    self.targets = torch.tensor(self.targets, dtype=torch.long) # Changed dtype to torch.long

  def __len__(self):
    return len(self.targets)  # Should match the number of targets, not individual rows

  def __getitem__(self, idx):
    return self.data[idx], self.targets[idx]

In [ ]:
# Create dataset
dataset = EEGDataset(epochs_data)

In [ ]:
import torch
import torch.nn as nn

def pos_encoding_for_one_position(x):
    """
    Returns [sin(x), cos(x), sin(x/2), cos(x/2)] of shape (4,)
    """
    x = torch.tensor(float(x))
    return torch.tensor([
        torch.sin(x),
        torch.cos(x),
        torch.sin(x / 2),
        torch.cos(x / 2),
    ])


def get_custom_pos_encoding(seq_len, d_model):
    """
    Make full positional encoding for shape:
    (1, seq_len, d_model)

    - Base PE is 4 dims
    - Repeat until reaching d_model
    """
    # (seq_len, 4)
    base = torch.stack([pos_encoding_for_one_position(i) for i in range(seq_len)], dim=0)

    repeat_times = d_model // 4
    remainder = d_model % 4

    # Repeat 4-d encoding to fill d_model
    full = base.repeat(1, repeat_times)     # (seq_len, 4 * repeat_times)

    if remainder > 0:
        full = torch.cat([full, base[:, :remainder]], dim=1)

    return full.unsqueeze(0)                 # (1, seq_len, d_model)

In [ ]:
class Block(nn.Module):
  def __init__(self, inplace):
    super().__init__()
    self.conv1 = nn.Conv1d(in_channels=inplace, out_channels=32, kernel_size=2, stride=2, padding=0)
    self.conv2 = nn.Conv1d(in_channels=inplace, out_channels=32, kernel_size=4, stride=2, padding=1)
    self.conv3 = nn.Conv1d(in_channels=inplace, out_channels=32, kernel_size=8, stride=2, padding=3)
    self.relu = nn.ReLU()

  def forward(self, x):
    x1 = self.relu(self.conv1(x))
    x2 = self.relu(self.conv2(x))
    x3 = self.relu(self.conv3(x))
    x = torch.cat([x1, x2, x3], dim=1)
    return x

In [ ]:
class CNNWithTransformerClassifier(nn.Module):
  def __init__(self, input_dim=14, num_classes=3, seq_len=64, d_model=96, nhead=4, num_layers=2):
    super().__init__()

    self.block1 = Block(input_dim)
    self.block2 = Block(96)
    self.block3 = Block(96)

    self.embedding = nn.Linear(96, d_model)
    # self.pos_encoding = nn.Parameter(torch.randn(1, seq_len, d_model))
    self.pos_encoding = nn.Parameter(get_custom_pos_encoding(seq_len, d_model), requires_grad=False)

    encoder_layer = nn.TransformerEncoderLayer(
      d_model=d_model,
      nhead=nhead,
      dim_feedforward=2 * d_model,
      dropout=0.2,
      batch_first=True
    )
    self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    self.classifier = nn.Sequential(
      nn.AdaptiveAvgPool1d(1),
      nn.Flatten(),
      nn.Linear(d_model, num_classes)
    )

  def forward(self, x):
    x = x.permute(0, 2, 1) # Add this line to permute the input tensor
    x = self.block1(x)
    x = self.block2(x)
    # x = self.block3(x)
    x = x.permute(0, 2, 1)
    x = self.embedding(x) + self.pos_encoding[:, :x.size(1), :]
    x = self.encoder(x)
    x = x.permute(0, 2, 1)
    return self.classifier(x)

In [ ]:
from pytorch_lightning.core.module import LightningModule # Added this import

class LitTransformer(LightningModule):
    def __init__(self, input_dim=14, num_classes=3, seq_len=64, lr=1e-5):
        super().__init__()
        self.save_hyperparameters()
        self.model = CNNWithTransformerClassifier(input_dim, num_classes, seq_len)
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        X, y = batch
        logits = self(X)
        loss = self.criterion(logits, y)
        preds = logits.argmax(dim=1)
        acc = (preds == y).float().mean()

        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('train_acc', acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def on_train_epoch_end(self):
        train_loss = self.trainer.callback_metrics.get("train_loss")
        train_acc = self.trainer.callback_metrics.get("train_acc")
        # Removed print statement to avoid RecursionError
        # if train_loss is not None and train_acc is not None:
        #     print(f"\nEpoch {self.current_epoch + 1}: train_loss={train_loss:.4f}, train_acc={train_acc*100:.2f}%")


    def validation_step(self, batch, batch_idx):
        X, y = batch
        logits = self(X)
        loss = self.criterion(logits, y)
        preds = logits.argmax(dim=1)
        acc = (preds == y).float().mean()

        self.log('val_loss', loss, on_epoch=True, prog_bar=True)
        self.log('val_acc', acc, on_epoch=True, prog_bar=True)
        return {'val_loss': loss, 'val_acc': acc}

    def on_validation_epoch_end(self):
        val_loss = self.trainer.callback_metrics.get("val_loss")
        val_acc = self.trainer.callback_metrics.get("val_acc")
        # Removed print statement to avoid RecursionError
        # if val_loss is not None and val_acc is not None:
        #     print(f"Epoch {self.current_epoch + 1}: val_loss={val_loss:.4f}, val_acc={val_acc*100:.2f}%")


    def test_step(self, batch, batch_idx):
        X, y = batch
        logits = self(X)
        preds = logits.argmax(dim=1)
        acc = (preds == y).float().mean()

        # log test accuracy
        self.log('test_acc', acc, on_epoch=True, prog_bar=True)
        return {'test_acc': acc}


    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

In [ ]:
train_size = int(0.7 * len(dataset))  # 80% cho training
val_size = int(0.20 * len(dataset))  # 10% cho validation
test_size = len(dataset) - train_size - val_size # 10% cho validation

# Chia dataset thành train và val
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size], generator=generator)

# Tạo DataLoader cho train và val
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
model = LitTransformer(
    input_dim=14,
    num_classes=3,  # Changed from 2 to 3 to accommodate DROWNSY_CLASS
    seq_len=64,
    lr=1e-5
)

In [ ]:
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import CSVLogger
import pytorch_lightning as pl

In [ ]:
logger = CSVLogger("logs", name="my_model")
trainer = Trainer(
    max_epochs=50,
    logger=logger,
    accelerator='auto',
    devices=1,
)

In [ ]:
trainer.fit(model, train_loader, val_loader)

In [ ]:
trainer.test(model, dataloaders=test_loader)

In [ ]:
metrics_file = f"{logger.log_dir}/metrics.csv"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd

df = pd.read_csv(metrics_file)
train_losses_plot = df['train_loss'].values  # skip first epoch
val_losses_plot = df['val_loss'].values
train_accs_plot = df['train_acc'].values
val_accs_plot = df['val_acc'].values

train_losses_plot = df['train_loss'].dropna().values
val_losses_plot = df['val_loss'].dropna().values
train_accs_plot = df['train_acc'].dropna().values
val_accs_plot = df['val_acc'].dropna().values

print(len(train_losses_plot), len(val_losses_plot), len(train_accs_plot), len(val_accs_plot))

In [ ]:
import matplotlib.pyplot as plt

# ------------------------------
# Ensure lists exist
# ------------------------------
train_losses = train_losses_plot
val_losses = val_losses_plot
train_accs = train_accs_plot
val_accs = val_accs_plot

# ------------------------------
# LOSS plot
# ------------------------------
min_len_loss = min(len(train_losses), len(val_losses))
epochs_loss = range(1, min_len_loss + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_loss, train_losses[:min_len_loss], label='Train Loss', marker='o')
plt.plot(epochs_loss, val_losses[:min_len_loss], label='Validation Loss', marker='o')
plt.title('Epoch vs Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# ------------------------------
# ACCURACY plot
# ------------------------------
min_len_acc = min(len(train_accs), len(val_accs))
epochs_acc = range(1, min_len_acc + 1)

plt.subplot(1, 2, 2)
plt.plot(epochs_acc, [x * 100 for x in train_accs[:min_len_acc]],
         label='Train Accuracy', marker='o')
plt.plot(epochs_acc, [x * 100 for x in val_accs[:min_len_acc]],
         label='Validation Accuracy', marker='o', color='green')
plt.title('Epoch vs Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()